# 03. Iterators and generators

**Goal.** Process data of any size in constant memory, and know what the interpreter does on each `for` step.

**You will be able to:**
1. Implement the iterator protocol and explain why a second pass over an iterator is empty.
2. Describe how a generator suspends and resumes a frame.
3. Build lazy pipelines for log files and batch jobs with `itertools`.
4. Measure the memory and latency cost of lazy against eager code.

## 1. The iterator protocol

### How it works under the hood
- `iter(obj)` calls `type(obj).__iter__`. The result must implement `__next__`. An iterator also returns itself from `__iter__`.
- A `for` loop calls `iter` once, then `__next__` until `StopIteration`. In CPython the loop is a `FOR_ITER` bytecode that calls the iterator's C slot `tp_iternext` directly.
- An iterable (list, dict, file) creates a fresh iterator on each `iter` call. An iterator is one-shot state: it remembers a position and cannot rewind.
- `iter(callable, sentinel)` builds an iterator that calls `callable()` until it returns `sentinel`.

### Production use
- Database drivers return cursors that fetch rows in pages. Iterating a cursor streams a result set larger than memory.
- Files iterate line by line. Reading a 40 GB log with `for line in f` uses one buffer, not 40 GB.
- Binary protocols read fixed-size blocks with `iter(lambda: f.read(8192), b"")`.

### Trade-offs

| Style | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Iterator class (`__iter__`, `__next__`) | One object with position state | Python-level call per item | Verbose | Full control, can carry extra methods | Boilerplate, easy to get wrong |
| Iterable returning fresh iterators | Container only | Fast re-iteration | Natural for collections | Safe to loop twice | Needs a stored collection |
| Single-use iterator | Constant | Cheapest per item | Surprises on second pass | Streams without storing | Silent empty second loop |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "understanding_python").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("python", sys.version.split()[0], "| package root", ROOT.name)

python 3.13.6 | package root understanding-python


In [2]:
import io

from understanding_python.functional import Countdown

nums = [10, 20, 30]
it = iter(nums)
print(type(it).__name__, "| iterator returns itself:", iter(it) is it)
print("manual steps:", next(it), next(it), next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration after the last item")

it = iter(nums)
manual = []
while True:
    try:
        manual.append(next(it))
    except StopIteration:
        break
print("for-loop desugared:", manual == [x for x in nums])

cd = Countdown(3)
print("first pass :", list(cd))
print("second pass:", list(cd), "(iterator is spent)")


class Batches:
    """Iterable: every `iter()` call starts a fresh pass."""

    def __init__(self, items, size):
        self.items, self.size = items, size

    def __iter__(self):
        for i in range(0, len(self.items), self.size):
            yield self.items[i:i + self.size]


b = Batches(list(range(7)), 3)
print("iterable twice:", list(b), list(b))

stream = io.BytesIO(b"x" * 20_000)
blocks = list(iter(lambda: stream.read(8192), b""))
print("block sizes via iter(callable, sentinel):", [len(blk) for blk in blocks])

list_iterator | iterator returns itself: True
manual steps: 10 20 30
StopIteration after the last item
for-loop desugared: True
first pass : [3, 2, 1]
second pass: [] (iterator is spent)
iterable twice: [[0, 1, 2], [3, 4, 5], [6]] [[0, 1, 2], [3, 4, 5], [6]]
block sizes via iter(callable, sentinel): [8192, 8192, 3616]


## 2. Generators: functions that pause

### How it works under the hood
- A function containing `yield` returns a generator object when called. The body does not run yet.
- The generator owns a heap-allocated frame holding locals and the instruction pointer (`gi_frame.f_lasti`). `next()` resumes the frame until the next `yield`, which saves state and returns the value.
- `send(x)` resumes and makes the `yield` expression evaluate to `x`. `close()` raises `GeneratorExit` at the pause point, so `finally` blocks run.
- `return value` inside a generator raises `StopIteration(value)`. `yield from sub` delegates to a sub-generator and receives that value.
- A suspended generator costs one frame (a few hundred bytes), whatever the sequence length.

### Production use
- Streaming ETL: read, transform and write one record at a time.
- Infinite or very long sequences: ID generators, retry back-off schedules, tail-follow of a growing file.
- Coroutine-style accumulators (`send`) were the base of `async`/`await`. Today `asyncio` uses native coroutines built on the same frame mechanism.

### Trade-offs

| Style | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Generator function | One frame | Small resume cost per item | Linear code, state in locals | Lazy, composable, tiny | One pass only, no `len` or indexing |
| List comprehension | O(n) items | Fastest for small n | Compact | Random access, reusable | Whole result in memory |
| Hand-written iterator class | One object | Slightly slower | Verbose | Explicit state | Much more code |

In [3]:
import inspect
import sys

from understanding_python.functional import fib_gen, running_mean


def read_batches(rows, size):
    """Yield lists of rows. Report the total through the return value."""
    sent = 0
    try:
        for i in range(0, len(rows), size):
            batch = rows[i:i + size]
            sent += len(batch)
            yield batch
    finally:
        print(f"  [read_batches closing after {sent} rows]")
    return sent


gen = read_batches(list(range(10)), 4)
print("state before first next:", inspect.getgeneratorstate(gen))
print("first batch            :", next(gen), "| f_lasti:", gen.gi_frame.f_lasti, "| locals:", sorted(gen.gi_frame.f_locals))
print("state after yield      :", inspect.getgeneratorstate(gen))
print("rest                   :", list(gen))
print("state at end           :", inspect.getgeneratorstate(gen))

early = read_batches(list(range(100)), 10)
next(early)
early.close()
print("after close:", inspect.getgeneratorstate(early))


def flatten_orders(orders):
    for o in orders:
        yield from o["lines"]


orders = [{"id": 1, "lines": ["pen", "pad"]}, {"id": 2, "lines": []}, {"id": 3, "lines": ["ink"]}]
print("yield from:", list(flatten_orders(orders)))

avg = running_mean()
next(avg)
print("send() running mean of 4, 8, 6:", [avg.send(v) for v in (4, 8, 6)])

squares_list = [n * n for n in range(100_000)]
squares_gen = (n * n for n in range(100_000))
print(f"list of 100k squares: {sys.getsizeof(squares_list):,} bytes | generator: {sys.getsizeof(squares_gen)} bytes")
print("fib below 100:", list(fib_gen(100)))

state before first next: GEN_CREATED
first batch            : [0, 1, 2, 3] | f_lasti: 106 | locals: ['batch', 'i', 'rows', 'sent', 'size']
state after yield      : GEN_SUSPENDED
  [read_batches closing after 10 rows]
rest                   : [[4, 5, 6, 7], [8, 9]]
state at end           : GEN_CLOSED
  [read_batches closing after 10 rows]
after close: GEN_CLOSED
yield from: ['pen', 'pad', 'ink']
send() running mean of 4, 8, 6: [4.0, 6.0, 6.0]
list of 100k squares: 800,984 bytes | generator: 200 bytes
fib below 100: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


## 3. Lazy pipelines with `itertools`

### How it works under the hood
- Each stage is a generator that pulls from the previous one. Pulling one item from the last stage pulls one item through all stages. Nothing is stored between them.
- `itertools` functions (`islice`, `chain`, `groupby`, `batched`, `accumulate`) are implemented in C and hold only a few pointers.
- `groupby` groups consecutive equal keys only, so sort first or use a dictionary.
- `tee` buffers items between its copies. If one copy runs far ahead, the buffer grows without bound.

### Production use
- Parse, filter and aggregate multi-gigabyte logs on a laptop.
- Batch rows for bulk `INSERT`, API calls or S3 uploads with `itertools.batched` (3.12+) or `chunked`.
- Merge sorted shards with `heapq.merge`, also lazy.

### Trade-offs

| Style | Memory | Latency | Readability | Advantages | Disadvantages |
|---|---|---|---|---|---|
| Lazy pipeline | Constant, one item in flight | First result immediately | Reads like a data flow | Handles unbounded input | Errors surface late, one pass |
| Eager lists between stages | O(n) per stage | Whole stage before the next | Easy to debug | Inspect each stage | Memory grows with input |
| `tee` / re-reading | Buffer or re-open cost | Extra pass | Looks harmless | Two consumers | Buffer can balloon |

In [4]:
import itertools
import random
import time
import tracemalloc
from collections import Counter

from understanding_python.functional import chunked, log_records

SERVICES = ["payments", "search", "auth", "checkout", "inventory"]
LEVELS = ["INFO"] * 17 + ["WARN"] * 2 + ["ERROR"]


def make_log_lines(n, seed=42):
    """Deterministic fake access log: one line per call, a few malformed ones."""
    rng = random.Random(seed)
    for i in range(n):
        if i % 997 == 0:
            yield "-- rotated --\n"
            continue
        s = f"2026-02-03 {i // 3600 % 24:02d}:{i // 60 % 60:02d}:{i % 60:02d}"
        yield f"{s} {rng.choice(LEVELS)} {rng.choice(SERVICES)} request id={i} took={rng.randint(1, 900)}ms\n"


sample = list(itertools.islice(make_log_lines(5), 3))
print(*sample, sep="", end="")


def peak_kib(build):
    tracemalloc.start()
    result = build()
    peak = tracemalloc.get_traced_memory()[1] / 1024
    tracemalloc.stop()
    return result, peak


def eager(n):
    lines = list(make_log_lines(n))
    errors = [r for r in log_records(lines) if r["level"] == "ERROR"]
    return Counter(r["service"] for r in errors)


def lazy(n):
    return Counter(r["service"] for r in log_records(make_log_lines(n), level="ERROR"))


for name, fn in (("eager", eager), ("lazy", lazy)):
    _, peak = peak_kib(lambda: fn(30_000))
    t0 = time.perf_counter()
    counts = fn(200_000)
    print(f"{name:5} peak {peak:7.0f} KiB (30k lines) | {time.perf_counter() - t0:.2f} s (200k lines) | {dict(sorted(counts.items()))}")

errors = log_records(make_log_lines(3000), level="ERROR")
batches = chunked(errors, 50)
sizes = [len(b) for b in batches]
print("bulk-insert batches:", len(sizes), "| sizes (first 4):", sizes[:4], "| last:", sizes[-1])

rows = sorted(({"svc": r["service"], "ms": int(r["message"].split("took=")[1][:-2])}
               for r in log_records(make_log_lines(2000))), key=lambda r: r["svc"])
for svc, grp in itertools.groupby(rows, key=lambda r: r["svc"]):
    ms = [r["ms"] for r in grp]
    print(f"  {svc:9} calls={len(ms):4d} mean={sum(ms) / len(ms):6.1f} ms max={max(ms)} ms")

print("running total:", list(itertools.accumulate([5, 3, 8, 1])), "| batched:", list(itertools.batched("abcdefg", 3)))

-- rotated --
2026-02-03 00:00:01 INFO payments request id=1 took=760ms
2026-02-03 00:00:02 INFO search request id=2 took=229ms


eager peak    3818 KiB (30k lines) | 0.89 s (200k lines) | {'auth': 1991, 'checkout': 2069, 'inventory': 2017, 'payments': 2009, 'search': 1932}


lazy  peak      24 KiB (30k lines) | 0.70 s (200k lines) | {'auth': 1991, 'checkout': 2069, 'inventory': 2017, 'payments': 2009, 'search': 1932}
bulk-insert batches: 3 | sizes (first 4): [50, 50, 44] | last: 44
  auth      calls= 399 mean= 468.6 ms max=900 ms
  checkout  calls= 422 mean= 452.9 ms max=898 ms
  inventory calls= 410 mean= 420.6 ms max=898 ms
  payments  calls= 385 mean= 428.3 ms max=896 ms
  search    calls= 381 mean= 471.8 ms max=897 ms
running total: [5, 8, 16, 17] | batched: [('a', 'b', 'c'), ('d', 'e', 'f'), ('g',)]
